In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif

df = pd.read_csv("cleaned_dataset.csv")
print(df.shape)
df.head()


(891, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,65.6344,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S


Creates new features. New columns made from existing ones: family size, whether travelling alone, fare per person, title from the name, and age group.

In [2]:
# Family size = siblings/spouses + parents/children + the passenger
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

# 1 if the passenger travelled alone
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

# Ticket price shared across the family
df["FarePerPerson"] = df["Fare"] / df["FamilySize"]

# Title (Mr, Mrs, Miss...) pulled out of the name
df["Title"] = df["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
df["Title"] = df["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
df["Title"] = df["Title"].where(df["Title"].isin(["Mr", "Mrs", "Miss", "Master"]), "Rare")

# Group ages into categories
df["AgeGroup"] = pd.cut(df["Age"], bins=[0, 12, 18, 60, 100],
                        labels=["Child", "Teen", "Adult", "Senior"])

# Drop columns that are only identifiers
df = df.drop(columns=["PassengerId", "Name", "Ticket"])
df[["FamilySize", "IsAlone", "FarePerPerson", "Title", "AgeGroup"]].head()

,FamilySize,IsAlone,FarePerPerson,Title,AgeGroup
0,2,0,3.6250,Mr,Adult
1,2,0,32.8172,Mrs,Adult
2,1,1,7.9250,Miss,Adult
3,2,0,26.5500,Mrs,Adult
4,1,1,8.0500,Mr,Adult


Encoding. Models need numbers, so text columns are converted. Sex and AgeGroup use label (ordinal) encoding, because Sex has two values and age groups have a natural order. Embarked and Title use one-hot encoding, because their values have no order.

In [3]:
# Technique 1: label / ordinal encoding
df["Sex"] = df["Sex"].map({"female": 0, "male": 1})
df["AgeGroup"] = df["AgeGroup"].map({"Child": 0, "Teen": 1, "Adult": 2, "Senior": 3}).astype(int)

# Technique 2: one-hot encoding (one 0/1 column per category)
df = pd.get_dummies(df, columns=["Embarked", "Title"], drop_first=True, dtype=int)

print(df.shape)
df.head()

(891, 17)


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,FamilySize,IsAlone,FarePerPerson,AgeGroup,Embarked_Q,Embarked_S,Title_Miss,Title_Mr,Title_Mrs,Title_Rare
0,0,3,1,22.0,1,0,7.2500,2,0,3.6250,2,0,1,0,1,0,0
1,1,1,0,38.0,1,0,65.6344,2,0,32.8172,2,0,0,0,0,1,0
2,1,3,0,26.0,0,0,7.9250,1,1,7.9250,2,0,1,1,0,0,0
3,1,1,0,35.0,1,0,53.1000,2,0,26.5500,2,0,1,0,0,1,0
4,0,3,1,35.0,0,0,8.0500,1,1,8.0500,2,0,1,0,1,0,0


Scaling. Columns like Fare (0 to 66) and Age (0 to 80) use very different ranges. StandardScaler (mean 0, standard deviation 1) is used for Age and FarePerPerson, and MinMaxScaler (squeezes values into 0 to 1) is used for Fare and FamilySize.

In [4]:
print("Before scaling:")
print(df[["Age", "FarePerPerson", "Fare", "FamilySize"]].describe().round(2))

# Method 1: standardization
df["Age"] = StandardScaler().fit_transform(df[["Age"]])
df["FarePerPerson"] = StandardScaler().fit_transform(df[["FarePerPerson"]])

# Method 2: min-max scaling
df["Fare"] = MinMaxScaler().fit_transform(df[["Fare"]])
df["FamilySize"] = MinMaxScaler().fit_transform(df[["FamilySize"]])

print("After scaling:")
print(df[["Age", "FarePerPerson", "Fare", "FamilySize"]].describe().round(2))

Before scaling:
          Age  FarePerPerson    Fare  FamilySize
count  891.00         891.00  891.00      891.00
mean    29.36          15.14   24.05        1.90
std     13.02          14.29   20.48        1.61
min      0.42           0.00    0.00        1.00
25%     22.00           7.25    7.91        1.00
50%     28.00           8.30   14.45        1.00
75%     35.00          16.45   31.00        2.00
max     80.00          65.63   65.63       11.00
After scaling:
          Age  FarePerPerson    Fare  FamilySize
count  891.00         891.00  891.00      891.00
mean     0.00           0.00    0.37        0.09
std      1.00           1.00    0.31        0.16
min     -2.22          -1.06    0.00        0.00
25%     -0.57          -0.55    0.12        0.00
50%     -0.10          -0.48    0.22        0.00
75%      0.43           0.09    0.47        0.10
max      3.89           3.53    1.00        1.00


Feature selection. Two methods find the features most related to survival: correlation with the target, and a statistical test (ANOVA F-test) that keeps the 10 best.

$0

In [5]:
X = df.drop(columns="Survived")
y = df["Survived"]

# Method 1: correlation with Survived
print(df.corr()["Survived"].drop("Survived").sort_values(key=abs, ascending=False).round(2))

# Method 2: statistical test, keep the 10 best features
selector = SelectKBest(f_classif, k=10).fit(X, y)
scores = pd.Series(selector.scores_, index=X.columns).sort_values(ascending=False).round(1)
print(scores)

selected = list(X.columns[selector.get_support()])
print("Selected features:", selected)

Title_Mr        -0.55
Sex             -0.54
Title_Mrs        0.34
Pclass          -0.34
Title_Miss       0.34
Fare             0.32
FarePerPerson    0.27
IsAlone         -0.20
Embarked_S      -0.15
AgeGroup        -0.13
Parch            0.08
Age             -0.06
SibSp           -0.04
FamilySize       0.02
Title_Rare      -0.01
Embarked_Q       0.00
Name: Survived, dtype: float64
Title_Mr         383.9
Sex              372.4
Title_Mrs        117.7
Pclass           115.0
Title_Miss       112.9
Fare              99.6
FarePerPerson     70.1
IsAlone           38.4
Embarked_S        20.4
AgeGroup          14.7
Parch              6.0
Age                3.8
SibSp              1.1
FamilySize         0.2
Title_Rare         0.1
Embarked_Q         0.0
dtype: float64
Selected features: ['Pclass', 'Sex', 'Fare', 'IsAlone', 'FarePerPerson', 'AgeGroup', 'Embarked_S', 'Title_Miss', 'Title_Mr', 'Title_Mrs']


Save the updated dataset. The final model-ready dataset has no text columns and no missing values.$0

In [6]:
df.to_csv("updated_dataset.csv", index=False)
print(df.shape)

from google.colab import files
files.download("updated_dataset.csv")

(891, 17)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>